In [90]:
import pandas as pd
import numpy as np
import tqdm
from IPython.display import HTML

# Ch 2

## read

In [ ]:
df = pd.read_excel("data.xls", sheet_name = "HBAT_MISSING", na_values = ["NA", "."])
df[["V10", "V11", "V12", "V13", "V14"]] = df[["V10", "V11", "V12", "V13", "V14"]].astype("category")

## table 2-1

In [ ]:
# === 第一部分：資料摘要 (Summary Statistics of Missing Data) ===
# get indicators
n_cases = df.count()
mean_val = df.mean(numeric_only = True)
std_val = df.std(numeric_only = True)
miss_num = df.isnull().sum()
miss_pct = (df.isnull().sum() / len(df)) * 100

# get df
summary = pd.DataFrame({
    ("Number of Cases", ""): n_cases,
    ("Mean", ""): mean_val.round(1),
    ("Standard Deviation", ""): std_val.round(2),
    ("Missing Data", "Number"): miss_num,
    ("Missing Data", "Percent"): miss_pct.round(0)
})

summary = summary.reindex(df.columns)
summary = summary.drop("ID", axis = 0, errors = "ignore")
summary.columns = pd.MultiIndex.from_tuples(summary.columns)

print("=== Summary Statistics of Missing Data for original Sample ===")
display(summary)

# === 第二部分：案例摘要 (Summary of Cases) ===

case_summary = df.isnull().sum(axis = 1).value_counts().sort_index().to_frame("Number of Cases")

case_summary["Percent of Sample"] = (case_summary["Number of Cases"] / len(df)) * 100

case_summary.loc["Total"] = case_summary.sum()

case_summary.index.name = "Number of Missing Data per Case"
case_summary["Percent of Sample"] = case_summary["Percent of Sample"].round(1)

print("=== Summary of Cases ===")
display(case_summary)

=== Summary Statistics of Missing Data for original Sample ===


Number of Cases  Mean Standard Deviation Missing Data        
                                                   Number Percent
V1               49   4.0               0.93           21    30.0
V2               57   1.9               0.88           13    19.0
V3               53   8.1               1.41           17    24.0
V4               63   5.2               1.17            7    10.0
V5               61   2.9               0.78            9    13.0
V6               64   2.6               0.72            6     9.0
V7               61   6.8               1.68            9    13.0
V8               61  46.0               9.36            9    13.0
V9               63   4.8               0.83            7    10.0
V10              68   NaN                NaN            2     3.0
V11              68   NaN                NaN            2     3.0
V12              68   NaN                NaN            2     3.0
V13              69   NaN                NaN            1     1.0
V14              68   NaN                NaN            2     3.0

=== Summary of Cases ===


,Number of Cases,Percent of Sample
Number of Missing Data per Case,,
0,26.0,37.1
1,15.0,21.4
2,19.0,27.1
3,4.0,5.7
7,6.0,8.6
Total,70.0,100.0


## table 2-2

In [ ]:
missing_cases = df[df.isnull().any(axis = 1)].copy()

n_missing = missing_cases.isnull().sum(axis = 1)
pct_missing = (n_missing / len(df.columns[1: ])) * 100

patterns = missing_cases.isnull().map(lambda x: "S" if x else "")
patterns["ID"] = missing_cases["ID"]

# 合併所有資訊
table_2_2 = pd.DataFrame({
    "Case": missing_cases["ID"],
    "# Missing": n_missing,
    "% Missing": pct_missing.round(1)
})
table_2_2 = table_2_2.merge(patterns.rename(columns = {"ID": "Case"}), left_on = "Case", right_on = "Case")
table_2_2.set_index("Case", inplace = True)

# 排序
custom_order = [
    205, 202, 250, 255, 269, 238, 240, 253, 256, 259, 260, 228, 246, 
    225, 267, 222, 241, 229, 216, 218, 232, 248, 237, 249, 220, 213, 
    257, 203, 231, 219, 244, 227, 224, 268, 235, 204, 207, 221, 245, 
    233, 261, 210, 263, 214
]
table_2_2 = table_2_2.reindex(custom_order)

print("=== table_2_2 2.2: Patterns of Missing Data by Case ===")
display(table_2_2)

=== Table 2.2: Patterns of Missing Data by Case ===


,# Missing,% Missing,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14
Case,,,,,,,,,,,,,,,,
205,1,7.1,,,S,,,,,,,,,,,
202,2,14.3,S,,S,,,,,,,,,,,
250,2,14.3,S,,S,,,,,,,,,,,
255,2,14.3,S,,S,,,,,,,,,,,
269,2,14.3,S,,S,,,,,,,,,,,
238,1,7.1,S,,,,,,,,,,,,,
240,1,7.1,S,,,,,,,,,,,,,
253,1,7.1,S,,,,,,,,,,,,,
256,1,7.1,S,,,,,,,,,,,,,


## table 2-3

In [112]:
# === Table 2.3: Missing Data Patterns Summary (Exact Order) ===

# 1. 建立缺失模式 (這次我們用 tuple 來存變數名稱，方便後續比對)
# 例如: ("V1", "V3") 代表該列缺了 V1 和 V3
# 修改第 1 步：產生 Pattern 時強制排序
def get_pattern_tuple(row):
    missing_cols = row.index[row.isnull()]
    return tuple(missing_cols.sort_values())

# 產生每個案例的模式 tuple
pattern_tuples = df.apply(get_pattern_tuple, axis=1)

# 2. 統計每種模式的次數
pattern_counts = pattern_tuples.value_counts()

# 3. 建立表格
table_2_3 = pd.DataFrame({"Number of Cases": pattern_counts})

# 4. 產生 "X" 標記
# 建立一個跟 df 欄位一樣的空表格
pattern_display = pd.DataFrame("", index=table_2_3.index, columns=df.columns[1: ])
# 填入 X
for pattern in table_2_3.index:
    pattern_display.loc[[pattern], list(pattern)] = "X"

table_2_3 = pd.concat([table_2_3, pattern_display], axis=1)

# 5. 計算 Number of Complete Cases...
def calc_complete_cases(pattern):
    # pattern 是一個 tuple，例如 ("V1", "V3")
    # 移除這些欄位後，計算剩下的完整案例數
    return len(df.drop(columns=list(pattern)).dropna())

table_2_3["Number of Complete Cases..."] = [calc_complete_cases(p) for p in table_2_3.index]

# === 關鍵修改：手動指定排序順序 ===
# 這是依照你提供的圖片 (Table 2.3) 觀察出的順序
# 請注意：如果你的資料中沒有出現某個模式，Pandas 會自動忽略它
custom_order = [
    (),                                 # 1. 完全沒缺
    ("V3",),                            # 2. 缺 V3
    ("V1", "V3"),                       # 3. 缺 V1, V3
    ("V1",),                            # 4. 缺 V1
    ("V1", "V4"),                       # 5. 缺 V1, V4
    ("V4",),                            # 6. 缺 V4
    ("V3", "V4"),                       # 7. 缺 V3, V4
    ("V3", "V5"),                       # 8. 缺 V3, V5
    ("V4", "V5"),                       # 9. 缺 V4, V5
    ("V1", "V5"),                       # 10. 缺 V1, V5
    ("V1", "V2"),                       # 11. 缺 V1, V2
    ("V2",),                            # 12. 缺 V2
    ("V2", "V3"),                       # 13. 缺 V2, V3
    ("V2", "V7"),                       # 14. 缺 V2, V7
    ("V7",),                            # 15. 缺 V7
    ("V7", "V8"),                       # 16. 缺 V7, V8
    ("V8",),                            # 17. 缺 V8
    ("V2", "V8"),                       # 18. 缺 V2, V8
    ("V1", "V2", "V8"),                 # 19. 缺 V1, V2, V8
    ("V9",),                            # 20. 缺 V9
    ("V5", "V9"),                       # 21. 缺 V5, V9
    ("V1", "V3", "V7"),                 # 22. 缺 V1, V3, V7
    ("V1", "V3", "V5", "V8", "V12", "V13"),
    ("V2", "V3", "V5", "V6", "V9", "V12", "V13"),
    ("V2", "V3", "V6", "V7", "V8", "V9", "V11"),
    ("V3", "V4", "V5", "V6", "V7", "V8", "V9"),
    ("V2", "V3", "V4", "V5", "V6", "V7", "V9"),
    ("V1", "V3", "V4", "V5", "V6", "V9", "V12")
]

# 使用 reindex 強制排序
# 為了避免資料中有新模式不在列表裡而被丟掉，我們把沒在列表裡的加在後面
existing_patterns = table_2_3.index.tolist()
remaining_patterns = [p for p in existing_patterns if p not in custom_order]
final_order = [p for p in custom_order if p in existing_patterns] + remaining_patterns

table_2_3 = table_2_3.reindex(final_order)

# 6. 加上雙層表頭
new_cols = []
for col in table_2_3.columns:
    if col == "Number of Cases":
        new_cols.append(("Number<br>of Cases", ""))
    elif col == "Number of Complete Cases...":
        new_cols.append(("Number of Complete<br>Cases if Variables<br>Missing in Pattern Are<br>Not Used", ""))
    else:
        new_cols.append(("Missing Data Patterns", col))

table_2_3.columns = pd.MultiIndex.from_tuples(new_cols)
table_2_3.index = [""] * len(table_2_3)

print("=== Table 2.3: Missing Data Patterns Summary (Custom Sort) ===")
table_2_3 = table_2_3.to_html(escape=False)
display(HTML(table_2_3))

=== Table 2.3: Missing Data Patterns Summary (Custom Sort) ===
